In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors

### Brownian dynamics 

$$dX_t = -\nabla V(X_t) dt + \sqrt{2\beta^{-1}} dB_t$$

with potential

$$V(x) = (x_1^2-1)^2 + 2.0 * (x_1^2+x_2-1)^2$$ 

first, define $V$ and its gradient

In [ ]:
# a potential function
def V(X):
    return (X[0]**2 - 1)**2 + 2.0 * (X[0]**2 + X[1] - 1)**2

# gradient of potential function 
def gradV(X):
    return np.array(( 4.0 * X[0] * (X[0]**2 - 1.0 + 2.0*(X[0]**2 + X[1] - 1)), 4.0 * (X[0]**2 + X[1] - 1)) )

show the potential profile

In [ ]:
x = np.arange(-2.5, 2.5, 0.05)
y = np.arange(-2.5, 2.5, 0.05)
X, Y = np.meshgrid(x, y)

plt.figure(figsize = (6, 4))

contour_levels = [0.0, 1.0, 1.5, 2.0, 3.0, 4.0]

# evaluate potential on mesh
V_on_grid = V([X,Y])

fig = plt.figure(figsize=(7,4))
ax = fig.add_subplot(1, 1, 1)

# plot profile by pcolormesh
im = ax.pcolormesh(X, Y, V_on_grid, cmap='coolwarm',shading='auto', vmin=0, vmax=4)

# show contour lines
contours = ax.contour(X, Y, V_on_grid, contour_levels)
ax.clabel(contours, inline=True, fontsize=13,colors='black')

ax.set_aspect('equal')
ax.set_xlabel(r'$x_1$',fontsize=20)
ax.set_ylabel(r'$x_2$',fontsize=20, rotation=0)
ax.tick_params(axis='both', labelsize=20)

ax.set_xticks([-2.0, -1.0, 0, 1.0, 2.0])
ax.set_yticks([-2.0, -1.0, 0, 1.0, 2.0])
ax.set_xlim([-2.5, 2.5])
ax.set_ylim([-2.5, 2.5])

ax.set_title('V',fontsize=25)

# show colorbar
cbar = fig.colorbar(im, ax=ax, shrink=1.0)
cbar.ax.tick_params(labelsize=15)
plt.show()

### Compute mean by sampling SDE


We estimate the mean of a function $g$:

$$ \mathbb{E}_\pi(g) = \frac{1}{Z} \int_{\mathbb{R}^2} g(x) \mathrm{e}^{-\beta V(x)} dx,$$
where $\pi$ is the invariant density:

$$ \pi(x) = \frac{1}{Z} \mathrm{e}^{-\beta V(x)}.$$

### Method 1: 

By ergodic theorem, we have, with probability one (i.e. for almost all trajectories)

$$
 \lim_{T\rightarrow +\infty} \frac{1}{T} \int_0^T g(X_t) dt = \mathbb{E}_\pi(g)\,.
$$
where $X_t$ satisfies 
$$dX_t = -\nabla V(X_t) dt + \sqrt{2\beta^{-1}} dB_t.$$

Therefore, we can estimate the mean value of $g$ by computing the time average of $g$ along the process $X_t$.

In practice, we approximate the time average on the left hand side above using discrete time points with finite time interval $[0,T]$: 
$$
\lim_{T\rightarrow +\infty} \frac{1}{T} \int_0^T g(X_t) dt \approx \frac{1}{T} \int_0^T g(X_t) dt \approx \frac{1}{N} \sum_{n=1}^N g(X_n)
$$
where $X_n$ are states sampled from the process at time $t_n = nh$, and $h=\frac{T}{N}$.

We take $g(x_1,x_2)=x_1$ and estimate the mean using:

$$\mathbb{E}_\pi(g) = \frac{1}{T} \int_0^T g(X_t) dt \approx \frac{1}{N} \sum_{n=1}^N g(X_n)$$




In [ ]:
def g(X):
    return X[0]

# sample the SDE using Euler-Maruyama scheme
def sample(beta=1.0, N=10000, seed=42):
    rng = np.random.default_rng(seed=seed)
    X = [-1, 0]
    dim = 2 
    traj = [X]
    delta_t = 0.001
    save = 10
    g_sum = 0.0
    for i in range(N):
        b = rng.normal(size=(dim,))
        X = X - gradV(X) * delta_t + np.sqrt(2 * delta_t/beta) * b
        g_sum = g_sum + g(X) 
        if i % save==0:
            traj.append(X)

    return np.array(traj), g_sum / N

seed_list = [1, 199, 235, 37, 42]
# for each seed, generate a long trajectory 
for seed in seed_list:
    trajectory, g_mean = sample(beta=0.7, N=2000000, seed=seed)
    print (r'seed=%d, mean of g: %.4f' % (seed, g_mean))
    

### Method 2:  
  Use the fact that $ p(x,t) \rightarrow \pi(x)$.

  Therefore 
  $$ \mathbb{E}_\pi(g) = \frac{1}{Z} \int_{\mathbb{R}^2} g(x) \mathrm{e}^{-\beta V(x)} dx = \lim_{t\rightarrow + \infty} \int_{\mathbb{R}^2} g(x) p(x,t) dx$$

In practice, we approximate $p(x,t)$ using particles. 

Algorithm:

1. generate $M$ initial states $X_0^{(1)}, X_0^{(2)}, \dots, X_0^{(M)}$.
2. for each initial state $X_0^{(i)}$, sample the SDE $$dX_t = -\nabla V(X_t) dt + \sqrt{2\beta^{-1}} dB_t$$ up to time $T$. Let the final state to be $X_T^{(i)}$.
3. estimate
   $$\mathbb{E}_\pi(g) = \lim_{t\rightarrow + \infty} \int_{\mathbb{R}^2} g(x) p(x,t) dx \approx \frac{1}{M} \sum_{i=1}^M g(X_T^{(i)}).$$

In [ ]:
# vectorized version 
def gradV(X):
    return np.stack(( 4.0 * X[:,0] * (X[:,0]**2 - 1.0 + 2.0*(X[:,0]**2 + X[:,1] - 1)), 4.0 * (X[:,0]**2 + X[:,1] - 1)), axis=1)
    
def g1(X):
    return X[:,0]

# sample the SDE starting for multiple initial states. 
def sample(x0, rng, beta=1.0, N=10000):

    X = x0
    n_traj = x0.shape[0]   
    delta_t = 0.001
    for i in range(N):
        b = rng.normal(size=(n_traj, 2))
        X = X - gradV(X) * delta_t + np.sqrt(2 * delta_t/beta) * b

    return X

# number of particles
n_traj = 1000

N = 100000

seed_list = [1, 199, 235, 37, 42]
# for each seed  
for seed in seed_list:
    rng = np.random.default_rng(seed=seed)
    x0 = rng.normal(size=(n_traj, 2))
    X = sample(x0, rng, beta=0.7, N=N)
    mean_g = np.mean(g1(X))
    print (f'seed={seed}, mean of g: {mean_g}')

### OU Process

Brownian dynamics:
$$dX_t = -\nabla V(X_t) dt + \sqrt{2\beta^{-1}} dB_t$$

Invariant density:
$$ \pi(x) = \frac{1}{Z} \mathrm{e}^{-\beta V(x)}$$

With the choice $V(x) = \frac{\kappa |x|^2}{2}$, we get the OU process:

$$dX_t = -\kappa X_t dt + \sqrt{2\beta^{-1}} dB_t$$

The invariant density is (Gaussian):
$$ \pi(x) = \Big(\frac{2\pi}{\beta\kappa}\Big)^{-\frac{d}{2}} \exp\Big(-\frac{\beta\kappa |x|^2}{2}\Big)$$. 

In [ ]:
# sample the SDE using Euler-Maruyama scheme
def sample(dim=1, beta=1.0, kappa=1.0, N=10000, seed=42):
    rng = np.random.default_rng(seed=seed)
    X = np.zeros(dim)
    traj = [X]
    delta_t = 0.001
    save = 5
    for i in range(N):
        b = rng.normal(size=(dim,))
        X = X - kappa * X * delta_t + np.sqrt(2 * delta_t/beta) * b
        if i % save==0:
            traj.append(X)

    return np.array(traj)

beta = 1.0
kappa = 1.0
# generate a long trajectory 
trajectory = sample(dim=1, beta=beta, kappa=kappa, N=1000000, seed=400)
   
print ("Number of states:", trajectory.shape[0])

# plot histogram. The first 40000 states are discarded  
count, bins, _ = plt.hist(trajectory[40000:], 50, density=True) 
# compute invariant density (gaussian) 
gaussian_pdf = 1.0/np.sqrt(2 * np.pi / (beta * kappa)) * np.exp(-beta * kappa * bins**2 / 2)

plt.plot(bins, gaussian_pdf, linewidth=2, color='r')

plt.show()